# 1kg_eur — close-pair screen over the GRM shards

Scans the finished GRM shards for pairs above a relatedness threshold and
writes (a) the pair list with each pair's `a_ij`, and (b) the union of
individuals involved, as a plink `--keep` list.

**Why this exists.** `07_relatedness_screen.ipynb` needs KING to separate
parent-offspring from full-sib pairs, because both sit at `a_ij ~= 0.5` and the
GRM cannot tell them apart. But KING over the whole cohort is O(N²) — at
N = 227,561 that is 2.6e10 pairs, and it does not finish in practical time.
The GRM cannot classify a close pair, yet it can cheaply say *which pairs are
worth asking about*. Roughly 10–12% of AoU participants have a 1st- or
2nd-degree relative, so taking the union of individuals in above-threshold
pairs should drop N by about an order of magnitude — and KING's cost falls with
the square of that.

**Threshold 0.2.** The screen only has to avoid false *negatives*: a missed PO
pair stays in the FS bin and contaminates exactly the quantity this is meant to
clean, whereas an extra individual costs only a slightly larger KING run. 0.2
also picks up most 2nd-degree pairs, which is useful if the `a_ij ~= 0.25` bin
ever needs the same treatment.

**The shards are read in place, over the gcsfuse mount — nothing is copied to
this VM.** The scan is a single sequential pass over each shard with no random
access, which is the access pattern gcsfuse handles well, and the only thing
held in memory is one block of rows. The cost is network read time, not disk.

**Prerequisites:** `06_grm_shards.ipynb` — all `N_SHARDS` shards present in
`03_grm/shards/`.

**Output** feeds `07_relatedness_screen.ipynb`, which classifies the 1st-degree
pairs and emits the `deg1_classified.tsv` that `grm_class_tool accumulate`
takes via `--pair-classes`.

## Config

In [ ]:
import glob
import os
import re

import numpy as np

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
# The gcsfuse mount point IS the bucket root, so a gs:// path under the bucket
# and a path under WORKSPACE_BUCKET differ only by the prefix.
PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

R_DIR      = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
SHARD_DIR  = f"{R_DIR}/03_grm/shards"

# Must match the run that produced the shards -- 06_grm_shards.ipynb used 20.
# A wrong value gives wrong row ranges; the entry-count reconciliation at the
# end of the full scan is what catches that, so do not skip it.
N_SHARDS = 20

THRESHOLD = 0.2

# Local scratch for the outputs only (a few MB) -- the shards stay in the bucket.
OUT_DIR = os.path.expanduser("~/scratch_close_pair_screen")
os.makedirs(OUT_DIR, exist_ok=True)

# How much of a shard to hold in memory at once. The scan is sequential, so
# this trades memory for syscall/round-trip overhead; 256 MB is comfortably
# past the point where gcsfuse read throughput plateaus.
BLOCK_BYTES = 256 * 1024**2

print(f"shards    : {SHARD_DIR}")
print(f"n_shards  : {N_SHARDS}")
print(f"threshold : {THRESHOLD}")
print(f"outputs   : {OUT_DIR}")

## Discover the shards

The filename prefix is whatever `--out` the shard job used, so the shards are
discovered by glob and the `--parallel` index `k` is parsed from the trailing
`.grm.bin.<k>` suffix plink appends, rather than assuming a naming scheme. The
`.grm.id` is shared by every shard (same sample set, same order), so any one of
them will do.

In [ ]:
shard_paths = {}
for path in sorted(glob.glob(f"{SHARD_DIR}/*.grm.bin*")):
    m = re.search(r"\.grm\.bin\.(\d+)$", path)
    assert m, f"cannot parse a --parallel index from {path!r}"
    k = int(m.group(1))
    assert k not in shard_paths, f"two files claim shard {k}: {shard_paths[k]}, {path}"
    shard_paths[k] = path

found = sorted(shard_paths)
assert found == list(range(1, N_SHARDS + 1)), (
    f"expected shards 1..{N_SHARDS}, found {found} — "
    f"is N_SHARDS right, or did 06_grm_shards.ipynb not finish?"
)

id_paths = sorted(glob.glob(f"{SHARD_DIR}/*.grm.id"))
assert id_paths, f"no .grm.id in {SHARD_DIR}"
GRM_ID_PATH = id_paths[0]

total_bytes = sum(os.path.getsize(p) for p in shard_paths.values())
print(f"{len(shard_paths)} shards, {total_bytes / 1024**3:.1f} GiB total")
print(f"grm.id: {GRM_ID_PATH}")
for k in (1, 2, N_SHARDS):
    print(f"  shard {k:>2}: {os.path.getsize(shard_paths[k]) / 1024**3:6.2f} GiB  "
          f"{os.path.basename(shard_paths[k])}")

## Load the sample IDs

Row index `i` of the GRM is line `i` of `.grm.id`. plink writes `FID = 0` for
every row of a `.grm.id` built without pedigree, so `IID` (the `person_id`)
carries all the information — which is why `grm_class_tool` keys its pair-class
file on IID alone. Both columns are kept here because plink `--keep` wants
them.

In [ ]:
with open(GRM_ID_PATH) as fh:
    ids = [tuple(line.split()[:2]) for line in fh if line.strip()]

N_IDS = len(ids)
iids = np.array([iid for _, iid in ids])
assert len(set(iids)) == N_IDS, "duplicate IIDs in .grm.id"

expected_entries = N_IDS * (N_IDS + 1) // 2
print(f"{N_IDS:,} individuals")
print(f"expected GRM entries: {expected_entries:,} "
      f"({expected_entries * 4 / 1024**3:.1f} GiB)")
print(f"distinct FIDs: {sorted({fid for fid, _ in ids})[:5]}")
print(f"first 3 rows: {ids[:3]}")

assert expected_entries * 4 == total_bytes, (
    f"shards hold {total_bytes / 4:,.0f} entries but {N_IDS:,} individuals imply "
    f"{expected_entries:,} — shards missing, truncated, or from a different run"
)
print("\nshard bytes reconcile with the .grm.id sample count")

## Row-range recovery

A `--parallel k N` shard is a contiguous band of GRM rows, but plink records
nothing about which rows: it picks the split by balancing *off-diagonal* pair
counts (`v*(v-1) >= target`), and the file simply stores each row's
off-diagonals followed by that row's single diagonal entry. The range therefore
has to be reconstructed from the shard's size in floats.

This is a direct port of `plink_parallel_row_start` / `try_row_start` /
`resolve_shard_range` in [`src/grm_class_tool.cpp`](../src/grm_class_tool.cpp)
(lines 404–452). Keeping it identical matters: the accumulate step maps the same
bytes to the same pairs, so a divergence here would mean this screen and the
accumulator disagree about who is who. The guessed start is verified by
requiring the row lengths to sum to *exactly* the observed float count, with a
small offset search around the guess for the rounding cases.

In [ ]:
def triangle_divide_off_diag(target):
    if target == 0:
        return 1
    v = int(np.sqrt(target) + 1.0) + 1
    v = max(v, 1)
    while v > 1 and (v - 1) * (v - 2) >= target:
        v -= 1
    while v * (v - 1) < target:
        v += 1
    return v


def plink_parallel_row_start(n_ids, parallel_idx, n_shards):
    """parallel_idx is 0-based."""
    target = (n_ids * (n_ids - 1) * parallel_idx) // n_shards
    v = triangle_divide_off_diag(target)
    return 0 if v == 1 else v


def try_row_start(candidate_start, n_ids, shard_floats):
    """Row index one past the last row, if the rows from candidate_start fit exactly."""
    consumed, i = 0, candidate_start
    while consumed < shard_floats and i < n_ids:
        consumed += i + 1
        i += 1
    return i if consumed == shard_floats else None


def resolve_shard_range(n_ids, k, n_shards, shard_floats):
    """(row_start, row_end_exclusive), verified against the shard's float count."""
    guess = plink_parallel_row_start(n_ids, k - 1, n_shards)
    for off in (0, -1, 1, -2, 2, -3, 3):
        cand = guess + off
        if 0 <= cand < n_ids:
            row_end = try_row_start(cand, n_ids, shard_floats)
            if row_end is not None:
                return cand, row_end
    raise RuntimeError(
        f"could not resolve row range: n_ids={n_ids} parallel={k}/{n_shards} "
        f"shard_floats={shard_floats} guess={guess}"
    )


def shard_range(k):
    nbytes = os.path.getsize(shard_paths[k])
    assert nbytes % 4 == 0, f"shard {k}: size {nbytes} is not a multiple of 4"
    return resolve_shard_range(N_IDS, k, N_SHARDS, nbytes // 4)


ranges = {k: shard_range(k) for k in sorted(shard_paths)}
covered = sum(end - start for start, end in ranges.values())
for k in (1, 2, N_SHARDS):
    start, end = ranges[k]
    print(f"  shard {k:>2}: rows [{start:>7,}, {end:>7,})  {end - start:>7,} rows")
assert covered == N_IDS, f"ranges cover {covered:,} rows, expected {N_IDS:,}"
print(f"\nall {N_SHARDS} ranges are contiguous and cover all {N_IDS:,} rows")

## The scan

One sequential pass per shard. For a block of rows the whole block is compared
against the threshold at once with NumPy and the flat hit positions are mapped
back to `(i, j)` by searching the row-start offsets — no Python loop over rows
or entries, which is what makes a ~100 GiB scan I/O-bound rather than
interpreter-bound.

Row `i` holds `i + 1` entries: off-diagonals at columns `0..i-1` and the
diagonal at column `i`. Only `j < i` is kept, so each pair is seen once and
self-relatedness is never reported.

In [ ]:
def scan_shard(k, threshold, block_bytes=BLOCK_BYTES):
    """Yields (i, j, a_ij) for off-diagonal entries above threshold in shard k."""
    row_start, row_end = ranges[k]
    hits_i, hits_j, hits_a = [], [], []
    consumed = 0

    with open(shard_paths[k], "rb") as fh:
        i = row_start
        while i < row_end:
            # Rows get longer down the matrix, so size each block by bytes.
            rows_in_block = max(1, block_bytes // (4 * (i + 1)))
            stop = min(i + rows_in_block, row_end)

            lengths = np.arange(i, stop, dtype=np.int64) + 1
            starts = np.concatenate(([0], np.cumsum(lengths)[:-1]))
            n_floats = int(lengths.sum())

            buf = fh.read(4 * n_floats)
            if len(buf) != 4 * n_floats:
                raise RuntimeError(
                    f"{shard_paths[k]}: short read at row {i} "
                    f"({len(buf)} of {4 * n_floats} bytes)"
                )
            consumed += n_floats
            block = np.frombuffer(buf, dtype=np.float32)

            flat = np.flatnonzero(block > threshold)
            if flat.size:
                local_row = np.searchsorted(starts, flat, side="right") - 1
                col = flat - starts[local_row]
                row = local_row + i
                offdiag = col < row              # col == row is the diagonal
                hits_i.append(row[offdiag])
                hits_j.append(col[offdiag])
                hits_a.append(block[flat[offdiag]])
            i = stop

    expected = sum(r + 1 for r in range(row_start, row_end))
    assert consumed == expected, (
        f"shard {k}: read {consumed:,} floats, rows [{row_start}, {row_end}) "
        f"imply {expected:,}"
    )
    if not hits_i:
        return np.empty(0, np.int64), np.empty(0, np.int64), np.empty(0, np.float32)
    return (np.concatenate(hits_i), np.concatenate(hits_j),
            np.concatenate(hits_a).astype(np.float32))

### Test one shard first

Shard `N_SHARDS` is the best single test: it has the fewest rows but each is
~`N_IDS` floats long, so it exercises the densest part of the triangle and the
largest block reads. Check the timing and the shape of the `a_ij` distribution
here **before** committing to the full scan.

Expect clustering near 0.25, 0.5 and possibly 1.0. A smooth continuum with no
structure means the row-range recovery is wrong — stop and investigate rather
than scanning the rest.

In [ ]:
import time

TEST_K = N_SHARDS
start, end = ranges[TEST_K]
t0 = time.monotonic()
ti, tj, ta = scan_shard(TEST_K, THRESHOLD)
elapsed = time.monotonic() - t0

shard_gib = os.path.getsize(shard_paths[TEST_K]) / 1024**3
print(f"shard {TEST_K}/{N_SHARDS}: rows [{start:,}, {end:,}), {shard_gib:.2f} GiB")
print(f"{elapsed:.0f}s at {shard_gib / max(elapsed, 1e-9):.2f} GiB/s")
print(f"-> full scan ~{elapsed * total_bytes / max(os.path.getsize(shard_paths[TEST_K]), 1) / 60:.0f} min")
print(f"\n{len(ta):,} pairs above {THRESHOLD}")

if len(ta):
    print(f"a_ij range: {ta.min():.3f} - {ta.max():.3f}\n")
    for lo, hi, label in [(0.2, 0.35, "2nd deg"), (0.35, 0.7, "1st deg"),
                          (0.7, 1.5, "dup/MZ"), (1.5, np.inf, "implausible")]:
        print(f"  [{lo}, {hi}) {label:>12}: {int(((ta >= lo) & (ta < hi)).sum()):,}")

### Full scan

If the estimate above is uncomfortably long, the shards can be staged to local
disk first (`gcloud storage cp` on the `gs://` paths, ~100 GiB, needs the disk)
and `SHARD_DIR` repointed — but the streaming read is the default precisely so
that isn't necessary.

In [ ]:
all_i, all_j, all_a = [], [], []
t0 = time.monotonic()

for k in sorted(shard_paths):
    hi, hj, ha = scan_shard(k, THRESHOLD)
    all_i.append(hi); all_j.append(hj); all_a.append(ha)
    done = sum(os.path.getsize(shard_paths[n]) for n in sorted(shard_paths) if n <= k)
    print(f"[{k:>2}/{N_SHARDS}] {len(ha):>7,} hits  "
          f"({sum(len(a) for a in all_a):>8,} total, "
          f"{time.monotonic() - t0:>5.0f}s, {100 * done / total_bytes:>3.0f}%)")

hit_i = np.concatenate(all_i)
hit_j = np.concatenate(all_j)
hit_a = np.concatenate(all_a)
print(f"\n{len(hit_a):,} pairs above {THRESHOLD} in {time.monotonic() - t0:.0f}s")

The per-shard read counts were each asserted against their row range inside
`scan_shard`, and the ranges were asserted to tile all `N_IDS` rows, so the
whole matrix has now provably been visited exactly once.

In [ ]:
# Belt and braces: the pair space actually examined must equal N*(N-1)/2.
pairs_examined = sum(
    sum(r for r in range(start, end)) for start, end in ranges.values()
)
expected_pairs = N_IDS * (N_IDS - 1) // 2
assert pairs_examined == expected_pairs, (
    f"examined {pairs_examined:,} pairs, expected {expected_pairs:,}"
)
print(f"{pairs_examined:,} pairs examined = N*(N-1)/2 — full coverage")
print(f"{len(hit_a):,} above threshold "
      f"({100 * len(hit_a) / expected_pairs:.5f}% of the pair space)")

## Distribution

In [ ]:
print(f"{len(hit_a):,} pairs, a_ij from {hit_a.min():.3f} to {hit_a.max():.3f}\n")
edges = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.55, 0.7, 0.9, 1.1, 2.0]
for lo, hi in zip(edges[:-1], edges[1:]):
    n = int(((hit_a >= lo) & (hit_a < hi)).sum())
    bar = "#" * min(60, int(60 * n / max(1, len(hit_a)) * 4))
    print(f"[{lo:.2f}, {hi:.2f}) {n:>8,}  {bar}")

deg1 = int((((hit_a >= 0.35) & (hit_a < 0.7))).sum())
print(f"\n1st-degree band (0.35-0.7): {deg1:,} pairs")
print("  -> these are the PO/FS mixture 07_relatedness_screen.ipynb has to split")

## Write the outputs

Two files. The pair list keeps `a_ij` so `07_relatedness_screen.ipynb` can
cross-check KING's kinship against the GRM (`a_ij` should track `2 * KINSHIP`);
the `--keep` list is the union of individuals involved, written as
`FID<TAB>IID` to match the panel `.fam`.

Both are participant-level and therefore **never** committed — they go to local
scratch and the bucket only.

In [ ]:
import pandas as pd

pairs = pd.DataFrame({
    "FID1": [ids[i][0] for i in hit_i],
    "IID1": [ids[i][1] for i in hit_i],
    "FID2": [ids[j][0] for j in hit_j],
    "IID2": [ids[j][1] for j in hit_j],
    "grm_a": hit_a,
})

pairs_path = os.path.join(OUT_DIR, f"candidate_pairs_gt{THRESHOLD}.tsv")
pairs.to_csv(pairs_path, sep="\t", index=False)

involved = sorted(set(zip(pairs["FID1"], pairs["IID1"]))
                  | set(zip(pairs["FID2"], pairs["IID2"])))
keep_path = os.path.join(OUT_DIR, f"candidate_individuals_gt{THRESHOLD}.keep")
with open(keep_path, "w") as fh:
    for fid, iid in involved:
        fh.write(f"{fid}\t{iid}\n")

print(f"{len(pairs):,} pairs       -> {pairs_path}")
print(f"{len(involved):,} individuals -> {keep_path}")
print(f"\ncohort reduction: {len(involved):,} / {N_IDS:,} = "
      f"{100 * len(involved) / N_IDS:.1f}%")
print(f"KING pair space:  {(len(involved) / N_IDS) ** 2 * 100:.3f}% of the full cohort")

In [ ]:
import subprocess

SCREEN_GS = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
             f"{PROJECT_DIR}/{RUN_DIR}/04_relatedness/screen")
for local in (pairs_path, keep_path):
    subprocess.run(["gcloud", "storage", "cp", local,
                    f"{SCREEN_GS}/{os.path.basename(local)}"], check=True)
    print(f"-> {SCREEN_GS}/{os.path.basename(local)}")
print("\nSet KEEP_GS in 07_relatedness_screen.ipynb to the .keep path above.")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser("~/repos/AOU-covariance/notebooks/06a_close_pair_screen.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/06a_close_pair_screen.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")